# Containerizing ML Applications with Docker

## 🎯 Learning Objectives

- Understand Docker concepts
- Build Docker images for ML apps
- Create multi-stage builds
- Optimize image size
- Use Docker Compose for multi-container apps

## Why Docker?

**Problems without Docker:**
- "Works on my machine" syndrome
- Dependency conflicts
- Environment inconsistencies
- Difficult deployment

**Docker benefits:**
- Consistent environments
- Easy deployment
- Isolation
- Reproducibility
- Scalability


## Docker Basics

### Key Concepts

- **Image**: Blueprint for containers (like a class)
- **Container**: Running instance of an image (like an object)
- **Dockerfile**: Instructions to build an image
- **Registry**: Storage for images (Docker Hub, ECR, etc.)

### Basic Commands

```bash
# Build image
docker build -t myapp:v1 .

# Run container
docker run -p 8000:8000 myapp:v1

# List containers
docker ps

# Stop container
docker stop <container_id>

# Remove container
docker rm <container_id>
```


## Simple ML API Dockerfile


In [ ]:
# Create a simple Dockerfile
dockerfile_content = '''FROM python:3.11-slim

WORKDIR /app

# Copy requirements
COPY requirements.txt .

# Install dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application
COPY . .

# Expose port
EXPOSE 8000

# Run application
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

# Save to file (in practice)
print("Dockerfile created:")
print(dockerfile_content)


## Requirements File


In [ ]:
requirements = '''fastapi==0.104.1
uvicorn[standard]==0.24.0
scikit-learn==1.3.2
joblib==1.3.2
numpy==1.26.2
pydantic==2.5.0
'''

print("requirements.txt:")
print(requirements)


## Building and Running

```bash
# Build the image
docker build -t ml-api:v1 .

# Run the container
docker run -d -p 8000:8000 --name ml-api ml-api:v1

# Check logs
docker logs ml-api

# Test the API
curl http://localhost:8000/health
```


## Multi-Stage Build (Optimized)


In [ ]:
optimized_dockerfile = '''# Stage 1: Build dependencies
FROM python:3.11-slim AS builder

WORKDIR /app

# Install build dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \
    gcc \
    g++ \
    && rm -rf /var/lib/apt/lists/*

# Install Python packages
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# Stage 2: Runtime
FROM python:3.11-slim

WORKDIR /app

# Copy only necessary files from builder
COPY --from=builder /root/.local /root/.local

# Copy application code
COPY main.py .
COPY models/ models/

# Update PATH
ENV PATH=/root/.local/bin:$PATH

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

print("Optimized Multi-Stage Dockerfile:")
print(optimized_dockerfile)
print("\nBenefits:")
print("- Smaller image size (no build tools in final image)")
print("- Faster deployment")
print("- Better security (fewer packages)")


## Docker Compose for Multi-Container Apps


In [ ]:
docker_compose = '''version: '3.8'

services:
  # ML API Service
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/models
      - REDIS_HOST=redis
    depends_on:
      - redis
    volumes:
      - ./models:/app/models
  
  # Redis for caching
  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
  
  # Monitoring
  prometheus:
    image: prom/prometheus:latest
    ports:
      - "9090:9090"
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml
'''

print("docker-compose.yml:")
print(docker_compose)
print("\nUsage:")
print("  docker-compose up -d     # Start all services")
print("  docker-compose down       # Stop all services")
print("  docker-compose logs -f   # View logs")


## API with Redis Caching


In [ ]:
from fastapi import FastAPI
import redis
import json
import hashlib

app = FastAPI()

# Connect to Redis
try:
    cache = redis.Redis(host='redis', port=6379, decode_responses=True)
    cache.ping()
    print("✓ Connected to Redis")
except:
    cache = None
    print("⚠️  Redis not available, running without cache")

def get_cache_key(data):
    """Generate cache key from input"""
    return hashlib.md5(json.dumps(data, sort_keys=True).encode()).hexdigest()

@app.post("/predict")
async def predict(features: dict):
    # Check cache
    if cache:
        cache_key = get_cache_key(features)
        cached_result = cache.get(cache_key)
        
        if cached_result:
            print("✓ Cache hit")
            return {"prediction": json.loads(cached_result), "cached": True}
    
    # Perform prediction (mock)
    result = {"class": 0, "confidence": 0.95}
    
    # Cache result (1 hour TTL)
    if cache:
        cache.setex(cache_key, 3600, json.dumps(result))
        print("✓ Result cached")
    
    return {"prediction": result, "cached": False}

print("API with caching ready")


## Best Practices

### Image Optimization

1. **Use specific base images**
   ```dockerfile
   FROM python:3.11-slim  # Not 'latest'
   ```

2. **Minimize layers**
   ```dockerfile
   # Bad (multiple layers)
   RUN pip install pandas
   RUN pip install numpy
   
   # Good (single layer)
   RUN pip install pandas numpy
   ```

3. **Use .dockerignore**
   ```
   __pycache__
   *.pyc
   .git
   .env
   tests/
   *.md
   ```

4. **Order matters (cache)**
   ```dockerfile
   # Copy requirements first (changes less often)
   COPY requirements.txt .
   RUN pip install -r requirements.txt
   
   # Copy code last (changes often)
   COPY . .
   ```


## Health Checks


In [ ]:
healthcheck_dockerfile = '''FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .

# Add health check
HEALTHCHECK --interval=30s --timeout=3s --start-period=40s --retries=3 \
  CMD curl -f http://localhost:8000/health || exit 1

EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

print("Dockerfile with health check:")
print(healthcheck_dockerfile)


## Deployment Workflow

### 1. Build
```bash
docker build -t myapp:v1.0 .
```

### 2. Test Locally
```bash
docker run -p 8000:8000 myapp:v1.0
curl http://localhost:8000/health
```

### 3. Tag for Registry
```bash
docker tag myapp:v1.0 myregistry.com/myapp:v1.0
```

### 4. Push to Registry
```bash
docker push myregistry.com/myapp:v1.0
```

### 5. Deploy
```bash
# Pull and run on production server
docker pull myregistry.com/myapp:v1.0
docker run -d -p 8000:8000 myregistry.com/myapp:v1.0
```


## Key Takeaways

✅ Docker ensures consistent environments
✅ Multi-stage builds reduce image size
✅ Docker Compose manages multi-container apps
✅ Caching speeds up predictions
✅ Health checks enable monitoring
✅ Follow best practices for production
